# PHASE 2: Data Cleaning for further EDA Analysis

In [ ]:
# Helper functions
import pandas as pd
import numpy as np
from pathlib import Path

def missing_info(df):
    missing_info = pd.DataFrame({
        'Missing Count': df.isna().sum(),
        'Percent': (df.isna().mean() * 100).round(2),
        'Data type': df.dtypes,
    })
    
    missing_info = missing_info[missing_info['Missing Count'] > 0]
    return missing_info.sort_values('Percent',ascending=False)

def is_duplicates(df):
    print(f'Total Duplicates: {df.duplicated().sum()}')

def dedup(df: pd.DataFrame) -> pd.DataFrame: # should be done 1st
    """
    Remove duplicate property-room records based on (id, occupancy).
    The first occurrence is retained.

        - Duplicate (id, occupancy) pairs
    """

    df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')
    return df

def drop_columns(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    """
    Drop columns that are not useful for modeling.

    If columns is not provided, the default set of columns
    identified during EDA will be removed.

    col = ['id', 'title', 'address', 'gate_closing_time', 'total_bathroom']
    """

    drop_cols = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
    if columns is None:
        columns = drop_cols

    df = df.drop(columns=columns)
    return df

def drop_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop rows that are not useful for modelling.
    
    this func() drops the reows which,
    - ~1% rows with missing values in key columns
    - rent with 0 or NaNs
    - occupancy is NaN
    """

    df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
    # Remove invalid/placeholder rents.
    # The minimum realistic PG rent in Chennai is well above 1000.
    df = df[df['rent'] >= 1000]

    return df

def fill_boolean_amenities(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIll missing boolean amenites with False
    then convert the columns to bool type
    """
    bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
    ]   

    # imputation for amenities
    df[bool_cols] = df[bool_cols].fillna(False).astype(bool)

    assert df[bool_cols].isna().sum().sum() == 0 # should be 0
    assert (df[bool_cols].dtypes == bool).all() # should show: bool

    return df

def impute_transit_score(df: pd.DataFrame) -> pd.DataFrame:
    # this is for fix the left influend skew-ness (should be done before imputation)
    df['transit_score'] = df['transit_score'].replace(-10, np.nan)

    # creating tag for msiing values rows
    df['transit_score_missing'] = df['transit_score'].isna().astype(int)

    # impute with local (locality) median
    df['transit_score'] = (
         
         df.groupby('locality')['transit_score']
        .transform(lambda x: x.fillna(x.median()))
    )

    # impute with global median (if local median is Nan)
    df['transit_score'] = df['transit_score'].fillna(df['transit_score'].median())

    return df

def impute_lifestyle_score(df: pd.DataFrame) -> pd.DataFrame:
    
   # imputation for lifestyle_score
    df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)  # creating tag for msiing values rows

    df['lifestyle_score'] = ( 
            df.groupby('locality')['lifestyle_score']
            .transform(lambda x: x.fillna(x.median()))
    )  # impute with local (locality) median

    # impute with global median (if local median is Nan)
    df['lifestyle_score'] = df['lifestyle_score'].fillna(df['lifestyle_score'].median()) # impute with global median (if local median is Nan)

    return df

def save_csv(df: pd.DataFrame):
    output_dir = Path('../Data/raw/processed/EDA')
    output_dir.mkdir(parents=True, exist_ok=True)

    # save cleaned dataset
    df.to_csv(output_dir / '01_eda_processed.csv', index=False)
    print("Dataset saved successfully!")

## 2.1 Drop Rows & Columns

In [3]:
df = pd.read_csv(r'D:\Hustle\Chennai-PG\Data\raw\chennai_pg_dataset.csv') 

In [4]:
missing_info(df)

,Missing Count,Percent,Data type
gate_closing_time,1476,82.78,str
lifestyle_score,900,50.48,float64
transit_score,900,50.48,float64
mess,834,46.78,object
cooking_allowed,833,46.72,object
wifi,830,46.55,object
power_backup,830,46.55,object
common_tv,829,46.49,object
refrigerator,829,46.49,object
room_cleaning,806,45.20,object


In [5]:
is_duplicates(df)

Total Duplicates: 162


In [5]:
df = dedup(df)
df = drop_columns(df)
df = drop_rows(df)
df = fill_boolean_amenities(df)
df = impute_transit_score(df)
df = impute_lifestyle_score(df)
save_csv(df)

Dataset saved successfully!


In [6]:
is_duplicates(df)

Total Duplicates: 0


In [7]:
missing_info(df)

,Missing Count,Percent,Data type
parking,144,9.79,str
